In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split

In [3]:
# Load WHO Life Expectancy Dataset
file_path = "Life Expectancy Data.csv"
df = pd.read_csv(file_path)

# Standardizing column names for consistency
df.columns = df.columns.str.strip()

# Selecting relevant features for regression
target_col = "Life expectancy"
features = [col for col in df.columns if col not in ["Country", "Year", "Status", target_col]]

# Handle missing values
imputer = SimpleImputer(strategy="mean")
df[features] = imputer.fit_transform(df[features])
df[target_col] = imputer.fit_transform(df[[target_col]])

# Normalize data
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Prepare data for regression
X = df[features].values
y = df[target_col].values

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(2350, 18)
(2350,)
(588, 18)
(588,)


In [9]:
# Feature Selection using Lasso Regression (Automatic Feature Importance Selection)
lasso = LassoCV(cv=10, random_state=42)  # Lasso Cross-Validation to find the best alpha
lasso.fit(X_train, y_train)

# Extract feature importance
feature_importance = np.abs(lasso.coef_)
feature_selection_df = pd.DataFrame({"Feature": features,"Importance": feature_importance}).sort_values(by="Importance", ascending=False)
feature_selection_df

,Feature,Importance
7,under-five deaths,9.128016
1,infant deaths,8.990583
0,Adult Mortality,2.639287
11,HIV/AIDS,2.348592
17,Schooling,2.130041
16,Income composition of resources,1.389066
10,Diphtheria,0.973383
6,BMI,0.787622
8,Polio,0.662862
12,GDP,0.469814


In [13]:
# Define the threshold for feature selection
importance_threshold = 0.5

selected_features = feature_selection_df[feature_selection_df["Importance"] > importance_threshold]["Feature"].tolist()
filtered_df = df[selected_features + [target_col]]

In [14]:
# Saving the new dataset
filtered_file_path = "Life_Expectancy_Selected_Features.csv"
filtered_df.to_csv(filtered_file_path, index=False)